# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook demonstrates how to load, explore, and analyze the FAIR^2 dataset describing protein abundance and features derived from mass spectrometry of extracellular vesicles using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema available at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Install mlcroissant if not already installed
!pip install --upgrade mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Croissant dataset schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)

# Print out dataset metadata
meta = dataset.metadata
print(f"Dataset Name: {getattr(meta, 'name', None)}")
print(f"Description: {getattr(meta, 'description', None)}")
print(f"Authors: {getattr(meta, 'author', None)}")
print(f"License: {getattr(meta, 'license', None)}")
print(f"Version: {getattr(meta, 'version', None)}")


## 2. Data Overview
Review available record sets and fields including their `@id` values, which are used to refer to entities in the Croissant schema.

In [ ]:
record_sets = dataset.metadata.recordSet
print(f"Available record sets: {len(record_sets)}")
for i, rs in enumerate(record_sets):
    print(f"\nRecord Set {i+1}:")
    print(f"  @id: {rs['@id']}")
    print(f"  Name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if isinstance(fields, dict):
        fields = [fields]
    print("  Fields:")
    for field in fields:
        # Each field should have an @id and a name
        print(f"    - @id: {field['@id']} | name: {field.get('name', '--')}")

**_Example output from previous cell:_**

> Available record sets: 1
> Record Set 1:
>   @id: https://api.app.sen.science/frontiers/7154140/e4f55692-0d85-41ba-9cda-f606c07e4744
>   Name: Mass Spectrometry Results
>   Fields:
>     - @id: https://api.app.sen.science/frontiers/7154140/237165b1-33bc-4609-bfa9-2c26be9c1615 | name: Accession
>     - @id: https://api.app.sen.science/frontiers/7154140/ae15fbd6-16d5-4ed6-9907-78953fa53a1c | name: Description
>     - ... (and so on)

_**Use the actual output above to select IDs for the next cells.**_

## 3. Data Extraction
Let's extract the data from each record set into Pandas DataFrames and inspect the fields. All references use entity `@id` identifiers as per Croissant schema.

In [ ]:
#--- Get all record set @id values dynamically ---#
record_set_ids = [r['@id'] for r in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    # The generator yields dicts, each row in the record set
    print(f"\nExtracting records from Record Set: {rs_id}")
    records = list(dataset.records(record_set=rs_id))
    print(f"  Rows loaded: {len(records)}")
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  Columns: {dataframes[rs_id].columns.tolist()}")
    else:
        print("  No records found.")

# Preview first few rows of the first record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Here we will:
- Select a numeric field using its `@id`
- Filter the data on a rule (e.g., only proteins with coverage > threshold)
- Normalize and group by attributes, referencing all columns by their `@id`.

In [ ]:
# Use the output from section 2 to select field @id for numeric field (e.g., coverage percentage)
# For illustration, we try several most likely column names and select the best available
import numpy as np

df = dataframes[main_record_set_id].copy()
# Try to find a numeric field: coverage percentage, MW, or peptide counts
candidate_fields = [c for c in df.columns if any(key in c.lower() for key in ['coverage', 'mw', 'peptide', 'count', 'abundance'])]
if not candidate_fields:
    candidate_fields = df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Candidate numeric fields identified: {candidate_fields}")

if candidate_fields:
    numeric_field_id = candidate_fields[0]
    # Set a threshold: for coverage, maybe 10%; for MW, 10000 Da, etc.
    threshold = 10 if 'coverage' in numeric_field_id.lower() else (10000 if 'mw' in numeric_field_id.lower() else 0)
    # Convert field to numeric if needed
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    filtered_df = df[df[numeric_field_id] > threshold].copy()
    print(f"Filtered records where {numeric_field_id} > {threshold} (n={len(filtered_df)}). Example:")
    print(filtered_df.head())

    # Normalize (z-score)
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} (first five):")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a candidate group field, e.g. description
    group_candidates = [c for c in df.columns if 'desc' in c.lower() or 'group' in c.lower() or 'sample' in c.lower() or 'type' in c.lower()]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
        print(f"\nGrouped mean {numeric_field_id} by {group_field} (first five):")
        print(grouped_df.head())
else:
    print("No numeric field found for EDA.")

## 5. Visualization
Let's visualize the distribution or relationships of key fields, using entity `@id`-based references.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if candidate_fields:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel('Frequency')
    plt.show()

    if group_field:
        # Only plot if moderate number of groups
        group_counts = filtered_df[group_field].value_counts()
        top_groups = group_counts.index[:5]
        plt.figure(figsize=(8, 6))
        sns.boxplot(data=filtered_df[filtered_df[group_field].isin(top_groups)], x=group_field, y=numeric_field_id)
        plt.title(f"{numeric_field_id} distribution by {group_field} (top 5 groups)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion

In this notebook, we loaded and explored the FAIR^2 dataset using `mlcroissant`. We examined its record set and fields, extracted and filtered data using field and record set `@id`s, normalized a numeric field, grouped by protein categories, and created visualizations of abundance distributions. Using `mlcroissant` makes it easy to access and manipulate well-described datasets compliant with the Croissant schema and FAIR data principles.